This module generally manages `Obsidian.md` vaults and their files and folders. 

The `VaultNote` class in this module is one of the most essentially classes used in `trouver`. Generally, one uses `Obsidian.md` to read and write "notes", which are `.md` files. The `VaultNote` class manages such notes in an `Obsidian.md` vault.

See Also `markdown.obsidian.personal.vault`.


In [ ]:
#| default_exp obsidian.vault

In [ ]:
#| export
from pathlib import Path
import os
from os import PathLike
from typing import Optional, Sequence, Union

from fastcore.basics import patch

from trouver.helper.files_and_folders import (
    path_no_ext, path_name_no_ext, md_files_in_dir
)
from trouver.obsidian.links import ObsidianLink, LinkType, replace_links_in_text

In [ ]:
import shutil
import tempfile
from unittest import mock

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import _test_directory
from trouver.obsidian.vault import _get_from_cache

Some examples for this module come from `nbs/_tests/vault_1`.

## Errors

In [ ]:
#| export
class NoteNotUniqueError(FileNotFoundError):
    """
    A `NoteNotUniqueError` is raised when a `VaultNote` is specified
    by name and not by relative path in the vault, but
    the vault is found to have multiple notes of the name.

    **Attributes**

    - `note_name` - str
        - The name of the note which is not unique in the vault.
    - `notes` - list[str]
        - The paths of the notes whose names are `note_name`.
    """
    def __init__(self, /, *args, **kwargs):
        super().__init__(*args, **kwargs)
            
    @classmethod
    def from_note_names(cls, note_name: str, notes: list[str]):
        """Construct a `NoteNotUniqueError` object from note names"""
        return cls(
            f'The note of the following name is not unique: {note_name}\n'\
            f'The name points to the following files: {notes}')


In [ ]:
show_doc(NoteNotUniqueError.from_note_names)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L37){target="_blank" style="float:right; font-size:smaller"}

### NoteNotUniqueError.from_note_names

>      NoteNotUniqueError.from_note_names (note_name:str, notes:list[str])

*Construct a `NoteNotUniqueError` object from note names*

In [ ]:
#| export
class NoteDoesNotExistError(FileNotFoundError):
    """
    A `NoteDoesNotExistError` is raised when a `VaultNote` is specified
    by either name and or by relative path in the vault, but
    the vault is found to have no notes of the name or path.
    """
    def __init__(self, /, *args, **kwargs):
        super().__init__(*args, **kwargs)

    @classmethod
    def from_note_name(
            cls,
            note_name: str):
        """Construct a `NoteDoesNotExistError` object from note name"""
        return cls(
            f'The note of the following name does not exist: {note_name}.'
            f' Make sure that the argument passed to `note_name` does not'
            f' erroneously end with `.md`, e.g. pass `this_is_a_note`'
            f' instead of `this_is_a_note.md`.')

In [ ]:
show_doc(NoteDoesNotExistError.from_note_name)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L55){target="_blank" style="float:right; font-size:smaller"}

### NoteDoesNotExistError.from_note_name

>      NoteDoesNotExistError.from_note_name (note_name:str)

*Construct a `NoteDoesNotExistError` object from note name*

In [ ]:
#| export
class NoteNotFoundInCacheError(RuntimeError):
    """
    A `NoteNotFoundInCacheError` is raised when a path corresponding to a
    `VaultNote` object is expected to be in the cache, but it is not.
    """
    def __init__(self, /, *args, **kwargs):
        super().__init__(*args, **kwargs)

    @classmethod
    def from_note_name(
            cls,
            note_name: str):
        """Construct a `NoteNotFoundInCacheError` object from note name"""
        return cls(
            f'The note of the following name does not exist: {note_name}.'
            f' Make sure that the argument passed to `note_name` does not'
            f' erroneously end with `.md`, e.g. pass `this_is_a_note`'
            f' instead of `this_is_a_note.md`.')
    

In [ ]:
show_doc(NoteNotFoundInCacheError.from_note_name)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L75){target="_blank" style="float:right; font-size:smaller"}

### NoteNotFoundInCacheError.from_note_name

>      NoteNotFoundInCacheError.from_note_name (note_name:str)

*Construct a `NoteNotFoundInCacheError` object from note name*

In [ ]:
#| export
class NotePathIsNotIdentifiedError(RuntimeError):
    """
    A `NotePathIsNotIdentifiedError` is raised when the `rel_path` attribute of a
    `VaultNote` object is expected to be identified (i.e. a path and not `None`) but
    this expectation is not fulfilled.
    """
    def __init__(self, /, *args, **kwargs):
        super().__init__(*args, **kwargs)

    @classmethod
    def from_note(
            cls,
            note):
        """Construct a `NotePatahIsNotIdentifiedErrro` object from a `VaultNote`."""
        return cls(f'The `rel_path` attribute of the `VaultNote` object is expected'
                   f' to be identified, and yet it is not. The vault of the'
                   f' `VaultNote` object is {note.vault} and the name of the object'
                   f' is {note.name}.')

In [ ]:
show_doc(NotePathIsNotIdentifiedError.from_note)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L97){target="_blank" style="float:right; font-size:smaller"}

### NotePathIsNotIdentifiedError.from_note

>      NotePathIsNotIdentifiedError.from_note (note)

*Construct a `NotePatahIsNotIdentifiedErrro` object from a `VaultNote`.*

## Convert path to Obsidian ID

In [ ]:
#| export
def path_to_obs_id(
        rel_path: PathLike # A path representation the path of an Obsidian note relative to its vault. This does not have to be an existing path.
        ) -> str: # The obsidian url of the hypothetical note within its vault. Note that this does not end with the file extension `.md`.
    """Convert a relative path of an Obsidian note to the Obsidian identifying
    str.
    
    This identification is for a vault-internal Wikilink.

    Note that this function does not have a vault as a parameter.
    """
    path_without_extension = path_no_ext(rel_path)
    return path_without_extension.replace('\\', '/')

Obsidian formats its file paths with '/'; the `path_to_obs_id` function converts a relative path of an Obsidian note to the Obsidian-recognized path.

In [ ]:
test_eq(path_to_obs_id(
    r'some_folder\some_subfolder\some_subsubfolder\some_file.md'),
    'some_folder/some_subfolder/some_subsubfolder/some_file')

Obsidian notes might contain spaces in their paths.

In [ ]:
test_eq(path_to_obs_id(
    Path(r'some folder\some subfolder\some file.md')),
    r'some folder/some subfolder/some file')

## Example vault


## Get all notes

In [ ]:
#| export
def all_paths_to_notes_in_vault(
        vault: PathLike,
        as_dict: bool = False
        ) -> Union[list[str], dict[str, list[str]]]:
    """Return the paths, relative to the Obsidian vault, of notes 
    in the Obsidian vault.
       
    **Parameters**

    - `vault` - `PathLike`
        - The path to the Obsidian vault directory
    - `as_dict` - `bool`
        - If `True`, then returns a dict. If `False`, then returns
        a list. Defaults to `False`. If there are multiple notes with the same
        name in the vault, and if `as_dict` is set to `True`, then the dictionary
        will contain a list of the (relative) paths (as str) to those notes.
        
        
    **Returns**

    - Union[list[str], dict[str, str]]
        - Each str represents the path relative to `vault`. If `as_dict` is
        True, then returns a dict whose keys are str, which are (unique) names
        of the notes in the vault, and the values are the paths.
    """
    return md_files_in_dir(dir=vault, root=vault, as_dict=as_dict)
    # vault = Path(vault)
    # paths =  [os.path.relpath(path, vault) for path in vault.glob(f'**/*.md')]
    # if as_dict:
    #     dicty = {path_name_no_ext(path): [] for path in paths}
    #     for path in paths:
    #         dicty[path_name_no_ext(path)].append(path)
    #     return dicty
    # else:
    #     return paths

The `all_paths_to_notes_in_vault` function returns all of the paths to notes in the Obsidian vault; only `.md` files are recognized as notes.

By default, the function returns a list whose items are strings of paths to notes relative to the vault path.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    # os.startfile(os.getcwd())

    notes = all_paths_to_notes_in_vault(temp_vault)
    test_eq(len(notes), 9)
    print(notes)
    # test_shuffled(, list)
    

['note_2_with_links_to_note_1.md', 'README.md', '_index.md', 'algebra\\ring.md', 'analysis\\exponential_function.md', 'category_theory\\category.md', 'topology\\category.md', 'topology\\note_1.md', 'algebra\\reference_1\\ring.md']


Passing `as_dict=True` returns a dictionary whose keys are note names and whose values are lists of paths to notes of the name relative to the vault.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    # os.startfile(os.getcwd())

    notes = all_paths_to_notes_in_vault(temp_vault, as_dict=True)
    test_eq(len(notes), 7)
    test_eq(len(notes['ring']), 2)
    print(notes)

{'note_2_with_links_to_note_1': ['note_2_with_links_to_note_1.md'], 'README': ['README.md'], '_index': ['_index.md'], 'ring': ['algebra\\ring.md', 'algebra\\reference_1\\ring.md'], 'exponential_function': ['analysis\\exponential_function.md'], 'category': ['category_theory\\category.md', 'topology\\category.md'], 'note_1': ['topology\\note_1.md']}


## Searching notes by name

In [ ]:
#| export
def all_note_paths_by_name(
        name: str,  # Name of the note(s) to find
        vault: PathLike,  # The path to the Obsidian vault directory
        subdirectory: Union[PathLike, None] = None # The path to a subdirectory in the Obsidian vault, relative to `vault`. If `None`, then denotes the root of the vault.
        ) -> list[Path]: # Each item is a path to a note of the given name, relative to `vault`.
    """Return the relative paths to all notes in the Obsidian vault 
    with the specified name in the specified subdirectory.
    
    This function does not assume that the specified subdirectory in the vault
    has at most one note of the specified name.
    
    """
    vault = vault if vault != None else ''
    vault = Path(vault)
    subdirectory = subdirectory if subdirectory != None else ''
    directory_path = vault / subdirectory
    all_notes_of_name = directory_path.glob(f'**/{name}.md')
    all_notes_of_name = list(all_notes_of_name)
    return [note_path.relative_to(vault) for note_path in all_notes_of_name]

Basic usage:

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    notes = all_note_paths_by_name('ring', temp_vault)
    print('Searched for notes of name `ring`:')
    print(notes, '\n')
    assert len(notes) == 2

    notes = all_note_paths_by_name('exponential_function', temp_vault)
    print('Searched for notes of name `exponential_function`:')
    print(notes, '\n')
    assert len(notes) == 1

    empty_list = all_note_paths_by_name('non-existent-note-name', temp_vault)
    print('Searching for a non-existent note name yields an empty list:')
    print(empty_list)
    assert len(empty_list) == 0

Searched for notes of name `ring`:
[Path('algebra/ring.md'), Path('algebra/reference_1/ring.md')] 

Searched for notes of name `exponential_function`:
[Path('analysis/exponential_function.md')] 

Searching for a non-existent note name yields an empty list:
[]


We can specify a subdirectory inside the vault to restrict the search to.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    all_notes_named_curve_in_vault = all_note_paths_by_name('category', temp_vault)
    print('All notes named `category`:\n', all_notes_named_curve_in_vault, '\n')
    assert len(all_notes_named_curve_in_vault) == 2
    for note_path in all_notes_named_curve_in_vault:
        test_eq(path_name_no_ext(note_path), 'category')
    
    print('All notes named `category` in the subdirectory `topology`')
    notes_named_topology_in_subdirectory = all_note_paths_by_name(
        'category', temp_vault, 'topology')
    print(notes_named_topology_in_subdirectory)
    assert len(notes_named_topology_in_subdirectory) == 1
    assert path_name_no_ext(notes_named_topology_in_subdirectory[0]) == 'category'

All notes named `category`:
 [Path('category_theory/category.md'), Path('topology/category.md')] 

All notes named `category` in the subdirectory `topology`
[Path('topology/category.md')]


In [ ]:
#| export
# TODO: include examples of `hints` parameter.
def note_path_by_name(
        name: str, # The path to the Obsidian vault directory.
        vault: PathLike, # The path to a subdirectory in the Obsidian vault. If `None`, then denotes the root of the vault.
        subdirectory: Union[PathLike, None] = None, # The path to a subdirectory in the Obsidian vault. If `None`, then denotes the root of the vault.
        hints: Union[list[PathLike], None] = None # Hints of which directories, relative to `subdirectory` that the note may likely be in. This is for speedup. The directories will be searched in the order listed.
        ) -> Path: # The note of the specified name in the specified subdirectory of the vault.
    """Return the path, relative to a subdirectory in the vault, 
    of the note of the specified name.

    **Raises**

    - NoteNotUniqueError
        - If the note of the specified name is not unique in the subdirectory.
    - NoteDoesNotExistError
        - If the note of the specified name does not exist in the subdirectory.

    **See Also**

    - The constructor of the `VaultNote` class
        - passing an argument to the `name` parameter of this constructor
        method essentially does the same thing as this function, except the
        constructor method uses a cache.
    """
    if not subdirectory:
        subdirectory = ''
    if not hints:
        hints = []
    vault = Path(vault)
    subdirectory = Path(subdirectory)
    # absolute_subdirectory = vault / subdirectory
    hints.append('')  # Search in subdirectory if all else fails
    for hint in hints:
        search_results = all_note_paths_by_name(name, vault, subdirectory / hint)
        if len(search_results) > 1:
            raise NoteNotUniqueError.from_note_names(name, search_results)
        elif len(search_results) == 1:
            return search_results[0]
    raise NoteDoesNotExistError.from_note_name(name)

Assuming that there exists a unique note of a specified name in a vault, we can identify it.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    note_path = note_path_by_name('exponential_function', temp_vault)
    print(note_path)
    test_eq(path_name_no_ext(note_path), 'exponential_function')

analysis\exponential_function.md


If there is more than one note in the vault of the specified name, then a `NoteNotUniqueError` is raised.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    print(f'The vault has more than one note named `category`:\n',
            all_note_paths_by_name('category', temp_vault))
    with (ExceptionExpected(ex=NoteNotUniqueError, regex='not unique')):
        note_path_by_name('category', temp_vault)

The vault has more than one note named `category`:
 [Path('category_theory/category.md'), Path('topology/category.md')]


Passing an argument to the parameter `subdirectory` restricts the search to the subdirectory. A `NoteNotUniqueError` can be avoided if a subdirectory is specified and if the note of the specified name is unique in the subdirectory.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    note = note_path_by_name('category', temp_vault, subdirectory='topology')
    print(note)
    assert 'topology' in str(note) and path_name_no_ext(note) == 'category'

topology\category.md


If there is no note in the vault of the specified name, then a `NoteDoesNotExistError` is raised.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    with (ExceptionExpected(ex=NoteDoesNotExistError, regex='does not exist')):
        note_path_by_name('this_note_does_not_exit', temp_vault)

The `hints` parameter

In [ ]:
#| export
def note_name_unique(
        name: str, # Name of the note.
        vault: PathLike # Path to the vault.
        ) -> bool: 
    """Return `True` if a note of the specified name exists and 
    is unique in the Obsidian vault.
    """
    return len(all_note_paths_by_name(name, vault)) == 1

Example use:

In [ ]:
# Example demonstration of the note_name_unique function
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    # Create a temporary vault for testing
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    # Check for a note that is known to exist and is unique
    unique_note_name = 'exponential_function'
    is_unique = note_name_unique(unique_note_name, temp_vault)
    print(f"Is the note '{unique_note_name}' unique in the vault? {is_unique}")
    assert is_unique, f"Expected '{unique_note_name}' to be unique, but it was not."

    # Check for a note that is known to exist but is not unique
    non_unique_note_name = 'category'
    print(f"Checking uniqueness for the note '{non_unique_note_name}'...")
    is_unique = note_name_unique(non_unique_note_name, temp_vault)
    print(f"Is the note '{non_unique_note_name}' unique in the vault? {is_unique}")
    assert not is_unique, f"Expected '{non_unique_note_name}' to not be unique, but it was."

    # Check for a note that does not exist
    nonexistent_note_name = 'this_note_does_not_exist'
    print(f"Checking uniqueness for the note '{nonexistent_note_name}'...")
    is_unique = note_name_unique(nonexistent_note_name, temp_vault)
    print(f"Is the note '{nonexistent_note_name}' unique in the vault? {is_unique}")
    assert not is_unique, f"Expected '{nonexistent_note_name}' to not be unique, but it was."

Is the note 'exponential_function' unique in the vault? True
Checking uniqueness for the note 'category'...
Is the note 'category' unique in the vault? False
Checking uniqueness for the note 'this_note_does_not_exist'...
Is the note 'this_note_does_not_exist' unique in the vault? False


## Getting note name from its path

In [ ]:
#| export
def note_name_from_path(
        note_path: str # The path of the note. The note does not need to exist.
        ) -> str: # The name of the note.
    """Return the name of a note from its path.
    """
    return path_name_no_ext(note_path)

In [ ]:
assert note_name_from_path('algebra/ring.md') == 'ring'